# Large Synthetic Conformational Graph Experiment

This notebook is a lightweight UI over `guided_conformational_path.large_synthetic`. The model and experiment definitions live in the Python module; this notebook only sets parameters, runs the experiment, and inspects results.

In [1]:
from pathlib import Path

from guided_conformational_path.large_synthetic import (
    display_py3dmol_path,
    LargeSyntheticConfig,
    PDBEndpointConfig,
    format_run_summary,
    route_family_costs,
    run_large_synthetic_experiment,
    run_pdb_endpoint_experiment,
    write_run_outputs,
)
from guided_conformational_path.visualization import plot_graph

## Parameters

Change these values to probe different graph sizes and sampling behavior. `n_residues` defaults to 200, but can be increased when runtime is acceptable.

In [2]:
config = LargeSyntheticConfig(
    n_residues=100,
    n_samples= 1500,
    noise_scale=0.15,
    seed=7,
    recursive_max_iterations=15,
    recursive_top_k=100,
    recursive_samples_per_parent=50,
    target_min_intermediates=20,
    target_max_intermediates=100,
    allow_fallback=False,
    fallback_waypoints=5,
    fallback_samples_per_waypoint=0,
    fallback_noise_scale=0.05,
    k_neighbors=0,
    outputs_dir=Path("outputs/notebook_large_synthetic"),
    write_nodes=False,
    write_pymol = True, 
    write_py3dmol = True
)
config


LargeSyntheticConfig(n_residues=100, n_samples=1500, noise_scale=0.15, seed=7, recursive_max_iterations=15, recursive_top_k=100, recursive_samples_per_parent=50, target_min_intermediates=20, target_max_intermediates=100, allow_fallback=False, fallback_waypoints=5, fallback_samples_per_waypoint=0, fallback_noise_scale=0.05, k_neighbors=0, outputs_dir=PosixPath('outputs/notebook_large_synthetic'), write_nodes=False, write_pymol=True, write_py3dmol=True, write_animation=True, animation_frames_per_edge=20, animation_fps=10, animation_dpi=120, graph_min_edge_weight=None, graph_max_edge_weight=None, graph_non_path_edge_alpha=0.125)

## Run Experiment

The experiment tries recursive Gaussian sampling from both endpoints first. It repeatedly generates noisy children from selected frontier nodes, keeps frontier parents from the closest cross-side A/B candidate pairs, and stops when it reaches `recursive_max_iterations` or the total `n_samples` cap. If that graph does not produce the requested path shape, it falls back to route-constrained waypoint families.


In [3]:
run = run_large_synthetic_experiment(config)
write_run_outputs(run, config)
print(format_run_summary(run, config.outputs_dir))

Residues: 100
Fallback used: False
Graph: 1502 nodes, 1127251 edges
Path intermediates: 4
Shortest path: A -> A_rec_040 -> A_rec_116 -> B_rec_051 -> B_rec_043 -> B
Total cost: 220.6164
Wrote outputs/notebook_large_synthetic/experiment_summary.json and outputs/notebook_large_synthetic/graph.png
Wrote path morph GIF: outputs/notebook_large_synthetic/path_morph.gif
Wrote PyMOL path view script: outputs/notebook_large_synthetic/pymol_path/view_path.pml
Wrote py3Dmol path view HTML: outputs/notebook_large_synthetic/py3dmol_path_view.html


## Inspect Summary Data

In [4]:
run.summary

{'fallback_used': False,
 'n_nodes': 1502,
 'n_edges': 1127251,
 'n_generated_nodes': 1500,
 'n_valid_nodes': 1502,
 'path_node_ids': ['A',
  'A_rec_040',
  'A_rec_116',
  'B_rec_051',
  'B_rec_043',
  'B'],
 'path_families': ['A_endpoint',
  'A_generated',
  'A_generated',
  'B_generated',
  'B_generated',
  'B_endpoint'],
 'path_intermediates': 4,
 'total_cost': 220.61639197105325,
 'route_family_costs': {},
 'generation_summaries': [{'phase': 'recursive_gaussian',
   'iteration': 1,
   'side': 'A',
   'parent': 'A',
   'requested': 50,
   'returned': 50,
   'rejected': 4,
   'attempted': 54,
   'max_attempts': 1000,
   'validity_enforced': True,
   'last_rejection_messages': ['1 consecutive C-alpha distances outside expected range'],
   'validity_settings': {'min_bond_distance': 2.5,
    'max_bond_distance': 4.5,
    'clash_distance': 2.0}},
  {'phase': 'recursive_gaussian',
   'iteration': 1,
   'side': 'B',
   'parent': 'B',
   'requested': 50,
   'returned': 50,
   'rejected': 6,

In [5]:
route_family_costs(run.graph)

{}

## Regenerate Graph Image

This is useful after editing `config.outputs_dir` or rerunning with different parameters.

In [6]:
plot_graph(run.graph, run.path_result.node_ids, config.outputs_dir / "graph.png")
config.outputs_dir / "graph.png"

PosixPath('outputs/notebook_large_synthetic/graph.png')

## What The Experiment Does

The large synthetic experiment is a controlled test of the graph pipeline.

It asks:

Can we generate candidate conformations, build a transition graph, and recover a lowest-cost path from A to B?

It runs in two stages.

## Stage 1: Recursive Gaussian Attempt

First, it creates two synthetic endpoint conformations:

A = starting C-alpha trace
B = target C-alpha trace

Then it uses `DummyPerturbationGenerator` recursively from both ends.

That means it does this:

A -> A_recursive frontier nodes -> deeper A-side nodes
B -> B_recursive frontier nodes -> deeper B-side nodes

After each expansion round, the sampler scores generated A-side nodes against generated B-side nodes and keeps frontier parents from the closest cross-side pairs.

If the recursive graph gives a useful path with the requested number of intermediates, it uses that.

## Stage 2: Fallback Waypoint Routes

If recursive Gaussian sampling does not produce a useful bridge, the experiment falls back to designed waypoint routes.

Default fallback creates 3 candidate route families:

low_energy
medium_detour
high_detour

Each route has 4 intermediate nodes by default:

A -> low_energy_01 -> low_energy_02 -> low_energy_03 -> low_energy_04 -> B
A -> medium_detour_01 -> ... -> medium_detour_04 -> B
A -> high_detour_01 -> ... -> high_detour_04 -> B

The graph then compares these route families by edge cost.

## Main Parameters

In the notebook or CLI:

```python
LargeSyntheticConfig(
    n_residues=200,
    n_samples=80,
    noise_scale=0.15,
    seed=7,
    recursive_max_iterations=6,
    recursive_top_k=8,
    recursive_samples_per_parent=4,
    target_min_intermediates=3,
    target_max_intermediates=4,
    fallback_waypoints=4,
    fallback_samples_per_waypoint=0,
    fallback_noise_scale=0.05,
    k_neighbors=0,
    outputs_dir=Path("outputs/notebook_large_synthetic"),
    write_nodes=False,
    write_pymol=True,
    write_py3dmol=True,
)
```

## Parameter Guide

| Parameter | Default | Description | Notes |
|-----------|---------|-------------|-------|
| `n_residues` | `200` | Number of C-alpha atoms in each synthetic protein conformation. | Increase to test larger graphs / structures. |
| `n_samples` | `80` | Total accepted recursive Gaussian node cap across both sides. | Prevents graph blow-up. |
| `noise_scale` | `0.15` | Amount of Gaussian coordinate noise to add. | Larger values increase node diversity but raise the chance of failing validity checks. |
| `seed` | — | Random seed for reproducibility. | |
| `recursive_max_iterations` | `6` | Maximum recursive expansion rounds. | Stops earlier if `n_samples` is exhausted. |
| `recursive_top_k` | `8` | Closest cross-side frontier parents kept per side. | Larger values increase frontier diversity and compute. |
| `recursive_samples_per_parent` | `4` | Gaussian children requested from each selected parent. | Actual accepted count can be lower after validity filtering. |
| `target_min_intermediates` | `3` | Minimum intermediate count that qualifies a recursive path as useful. | If no path falls within the min-max range, the fallback is triggered. |
| `target_max_intermediates` | `4` | Maximum intermediate count that qualifies a recursive path as useful. | Pair with `target_min_intermediates`. |
| `fallback_waypoints` | `4` | Number of designed intermediate nodes per fallback route family. | Produces paths of the form A -> wp1 -> ... -> wpn -> B. |
| `fallback_samples_per_waypoint` | `0` | Optional Gaussian samples placed around each fallback waypoint. | Keep at 0 for clarity. Higher values increase graph complexity. |
| `fallback_noise_scale` | — | Noise level for waypoint-centered Gaussian samples. | Only applies if `fallback_samples_per_waypoint > 0`. |
| `k_neighbors` | `0` | Controls graph sparsity. | `0` = full graph connectivity where applicable. |
| `outputs_dir` | — | Directory where output files are written. | |
| `write_nodes` | — | If `True`, writes all graph nodes as PDB-like files. | |
| `write_pymol` | — | If `True`, writes a PyMOL script to `pymol_path/view_path.pml`. | |
| `write_py3dmol` | — | If `True`, writes an interactive viewer to `py3dmol_path_view.html`. | |

## Output Files

Typical outputs:

```text
outputs/large_synthetic/
  experiment_summary.json
  path_summary.json
  graph.png
  pymol_path/
  py3dmol_path_view.html
```


# PDB Endpoint Experiment

Use this section when A and B are real endpoint PDB files rather than synthetic generated endpoints.

Place your files here:

```text
data/pdb_inputs/A.pdb
data/pdb_inputs/B.pdb
```

The loader reads C-alpha atoms. It first tries exact residue matching by chain/residue/insertion code, then can fall back to geometric matching if residue IDs do not align cleanly.

In [7]:
pdb_input_dir = Path("data/pdb_inputs")
pdb_input_dir.mkdir(parents=True, exist_ok=True)

endpoint_a_file = pdb_input_dir / "A.pdb"
endpoint_b_file = pdb_input_dir / "B.pdb"

print(f"Put endpoint A here: {endpoint_a_file} | exists={endpoint_a_file.exists()}")
print(f"Put endpoint B here: {endpoint_b_file} | exists={endpoint_b_file.exists()}")

Put endpoint A here: data/pdb_inputs/A.pdb | exists=False
Put endpoint B here: data/pdb_inputs/B.pdb | exists=False


In [8]:
pdb_config = PDBEndpointConfig(
    endpoint_a_path=endpoint_a_file,
    endpoint_b_path=endpoint_b_file,
    n_samples=80,
    noise_scale=0.15,
    seed=7,
    recursive_max_iterations=6,
    recursive_top_k=8,
    recursive_samples_per_parent=4,
    target_min_intermediates=3,
    target_max_intermediates=4,
    fallback_waypoints=4,
    fallback_samples_per_waypoint=0,
    fallback_noise_scale=0.05,
    k_neighbors=0,
    outputs_dir=Path("outputs/pdb_endpoint_experiment"),
    write_nodes=False,
    write_pymol=True,
    write_py3dmol=True,
    graph_min_edge_weight=None,
    graph_max_edge_weight=None,
    graph_non_path_edge_alpha=0.125,
    matching_method="exact_then_geometric",
    min_exact_match_fraction=0.8,
    geometric_match_distance_cutoff=5.0,
    min_geometric_matches=8,
)
pdb_config


PDBEndpointConfig(endpoint_a_path=PosixPath('data/pdb_inputs/A.pdb'), endpoint_b_path=PosixPath('data/pdb_inputs/B.pdb'), n_samples=80, noise_scale=0.15, seed=7, recursive_max_iterations=6, recursive_top_k=8, recursive_samples_per_parent=4, target_min_intermediates=3, target_max_intermediates=4, allow_fallback=True, fallback_waypoints=4, fallback_samples_per_waypoint=0, fallback_noise_scale=0.05, k_neighbors=0, outputs_dir=PosixPath('outputs/pdb_endpoint_experiment'), write_nodes=False, write_pymol=True, write_py3dmol=True, write_animation=True, animation_frames_per_edge=20, animation_fps=10, animation_dpi=120, graph_min_edge_weight=None, graph_max_edge_weight=None, graph_non_path_edge_alpha=0.125, matching_method='exact_then_geometric', min_exact_match_fraction=0.8, geometric_match_distance_cutoff=5.0, min_geometric_matches=8)

Run the PDB endpoint experiment. This uses recursive Gaussian noising from both loaded endpoints first; if that does not produce the requested path shape, it falls back to waypoint route families.


In [9]:
pdb_run = run_pdb_endpoint_experiment(pdb_config)
write_run_outputs(pdb_run, pdb_config)
print(format_run_summary(pdb_run, pdb_config.outputs_dir))
pdb_run.summary.get("input", {})

FileNotFoundError: [Errno 2] No such file or directory: 'data/pdb_inputs/A.pdb'

Display the PDB endpoint results. The graph image shows the abstract conformation graph; the py3Dmol viewer shows the selected path structures side-by-side.

In [ ]:
plot_graph(
    pdb_run.graph,
    pdb_run.path_result.node_ids,
    pdb_config.outputs_dir / "graph.png",
    min_edge_weight=pdb_config.graph_min_edge_weight,
    max_edge_weight=pdb_config.graph_max_edge_weight,
    non_path_edge_alpha=pdb_config.graph_non_path_edge_alpha,
)
display_py3dmol_path(pdb_run)

In [ ]:
{
    "summary": pdb_config.outputs_dir / "experiment_summary.json",
    "path_summary": pdb_config.outputs_dir / "path_summary.json",
    "networkx_graph": pdb_config.outputs_dir / "graph.png",
    "py3dmol_html": pdb_config.outputs_dir / "py3dmol_path_view.html",
    "pymol_script": pdb_config.outputs_dir / "pymol_path" / "view_path.pml",
}

In [ ]:
l = ["t", "", "s", "t"]
print("".join(l))